[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/18_troubleshooting_faq.ipynb)

# Troubleshooting & FAQ

> **Fine-tuning series — 18 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. Conceptual/reference — no GPU setup needed.


## Troubleshooting & FAQ

### Common errors → fixes

| Symptom | Likely cause | Fix |
|---|---|---|
| **CUDA out of memory (OOM)** | batch / seq / model too big | smaller `per_device_train_batch_size` + more `gradient_accumulation_steps`; gradient checkpointing; QLoRA; shorter `max_length` |
| **Loss is NaN / explodes** | LR too high, fp16 overflow | lower learning rate; use **bf16** not fp16; add warmup |
| **Loss won't go down** | LR too low, dead adapter, bad data | run the smoke test (02b); check `print_trainable_parameters() > 0`; raise LR; verify chat template |
| **`0%` trainable params** | wrong `target_modules` (fused QKV) | auto-discover linear layers (see 12); confirm names match your model |
| **Garbage / repetitive output** | template mismatch, over/under-trained, bad gen params | match the exact chat template; check epochs; set `temperature`/`top_p`; check EOS token |
| **Eval worse than base model** | overfitting or catastrophic forgetting | fewer epochs / lower rank; mix in general data; early stop |
| **`bitsandbytes` / Unsloth import error** | not on NVIDIA CUDA | those are CUDA-only; use plain LoRA on CPU/`mps`, or a CUDA GPU |
| **Training is very slow** | full FT / large model / no fused kernels | use LoRA/QLoRA; try Unsloth; reduce sequence length |

### Beginner FAQ

**Which model do I start with — base or instruct?**
Use an **instruct** model (e.g. `*-Instruct`) for chatbots, instruction following, and most
tasks — it already speaks in turns. Use a **base** model for *continued pretraining* on raw
domain text. Find models on the Hugging Face Hub; read the model card and **check the license**
(some restrict commercial use).

**How much data do I need?**
For a *behavior / format / tone* change: hundreds to a few thousand good examples. For
*substantial new knowledge*: far more — and often **RAG** is the better tool than fine-tuning.

**How many epochs?**
Usually **1–3**. Watch validation loss and stop when it stops improving (early stopping).

**How long / how much will it cost?**
LoRA/QLoRA on a small model + a few thousand examples is often **minutes to a couple of hours**
on one GPU. Rough cloud cost = GPU $/hr × hours (a T4-class GPU is cents–dollars/hr; an A100
more). QLoRA is cheapest because it fits on small GPUs.

**Where do I run it?**
Free **Google Colab** (T4) or **Kaggle** for small models; a rented **cloud A100/H100** for
bigger jobs. QLoRA makes consumer GPUs viable.

**Fine-tune, RAG, or just prompt?**
Prompt first (cheapest). Use **RAG** for fresh or citable *facts*. **Fine-tune** for new
*behavior, format, or skill*, lower latency/cost at scale, or alignment prompting can't reach.

**Will it forget its other abilities?**
Full fine-tuning can (**catastrophic forgetting**). LoRA/QLoRA reduce this by freezing the base;
mixing some general examples into your data helps further.

In [ ]:
# tiny symptom -> first-thing-to-try lookup (no dependencies)
FIXES = {
    "oom":            "smaller batch + grad accumulation; QLoRA; gradient checkpointing",
    "nan_loss":       "lower LR; use bf16 not fp16; add warmup",
    "loss_flat":      "smoke-test (02b); check trainable>0; raise LR; verify template",
    "zero_trainable": "wrong target_modules; auto-discover linear layers (12)",
    "garbage_output": "match chat template; tune temperature/top_p; check EOS",
}
for k, v in FIXES.items():
    print(f"{k:16s} -> {v}")